# 02-04_training-Gemma3-1B
<a href="https://colab.research.google.com/github/daehyun99/BurnFit-AI-developer-assignment/blob/main/data/02-04_training-Gemma3-1B.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 1. pip install

In [ ]:
!pip install -q gemma

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.7/5.7 MB 54.6 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 122.3/122.3 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 479.3/479.3 kB 20.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.4/55.4 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 400.4/400.4 kB 20.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.3/65.3 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 101.8/101.8 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.1/47.1 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.0/111.0 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.7/76.7 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.0/63.0 MB 19.5 MB/s eta 0

In [ ]:
!pip install -q openpyxl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 250.9/250.9 kB 4.9 MB/s eta 0:00:00


## 2. 세션 재시작 및 모델로드

In [ ]:
# prompt: google drive mount

from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
# Common imports
import os
import jax
import jax.numpy as jnp
import optax
import treescope
import json

# Gemma imports
from kauldron import kd
from gemma import gm
from gemma import peft

import pandas as pd

In [ ]:
os.environ["XLA_PYTHON_CLIENT_MEM_FRACTION"]="1.00"

In [ ]:
model = gm.nn.LoRA(
    rank=4,
    model=gm.nn.Gemma3_1B(tokens="batch.input"),
)

In [ ]:
token_ids = jnp.zeros((1, 256,), dtype=jnp.int32)  # Create the (batch_size, seq_length)

params = model.init(
    jax.random.key(0),
    token_ids,
)

params = params['params']

# 반드시 step 값 확인 필요!

In [ ]:
step = 6 # 값 확인!

In [ ]:
class MyInitTransform:
    def transform(self, state):
        original = gm.ckpts.load_params(gm.ckpts.CheckpointPath.GEMMA3_1B_IT)
        lora = gm.ckpts.load_params(f"/content/drive/MyDrive/Gemma3-lora{step-1}")
        merged = peft.merge_params(original, lora)
        return state.replace(params=merged)  # 중요! state에 params를 설정해야 함


In [ ]:
# # # 첫 로드인 경우에만
# init_transform = gm.ckpts.SkipLoRA(
#     wrapped=gm.ckpts.LoadCheckpoint(
#         path=gm.ckpts.CheckpointPath.GEMMA3_1B_IT,
#     ),
# )

In [ ]:
optimizer = kd.optim.partial_updates(
    optax.adafactor(learning_rate=0.005),
    # We only optimize the LoRA weights. The rest of the model is frozen.
    mask=kd.optim.select("lora"),
)

In [ ]:
tokenizer = gm.text.Gemma3Tokenizer()

tokenizer.encode('This is an example sentence', add_bos=True)

[<_Gemma3SpecialTokens.BOS: 2>, 2094, 563, 614, 2591, 13315]

## 3. 학습 데이터 로드

In [ ]:
df = pd.read_excel(f"/content/drive/MyDrive/05_[공유파일]/Burnfit_dataset/teacher_data.xlsx")

In [ ]:
json_data = [
    {
        "question": f"{row['instruction']} {row['input']}",
        "answer": row["output"]
    }
    for _, row in df.iterrows()
]

In [ ]:
with open("/content/drive/MyDrive/05_[공유파일]/Burnfit_dataset/teacher_data.json", "w", encoding="utf-8") as f:
    json.dump(json_data, f, indent=2, ensure_ascii=False)

In [ ]:
# data를 json = {question, answer 구조로 제작해서 json 형태로 만들기}
ds = kd.data.py.Json(
    "/content/drive/MyDrive/05_[공유파일]/Burnfit_dataset/teacher_data.json",
    shuffle=False,
    batch_size=8,
    transforms=[
        gm.data.Seq2SeqTask(
            in_prompt='question',
            in_response='answer',
            out_input='input',
            out_target='target',
            out_target_mask='loss_mask',
            tokenizer=tokenizer,
            max_length=1024,
            truncate=True
        ),
    ],
)

ex = ds[0]


In [ ]:
text = tokenizer.decode(ex['input'][0])

print(text)

<start_of_turn>user
요즘 허리가 안 좋아서 데드는 무리하지 않게 조절해줬으면 해. {'성별': '남성', '몸무게': 121.05, 'squat_1RM': 215.0, 'press_1RM': 103.2, 'bench_press_1RM': 147.5, 'deadlift_1RM': 222.5}<end_of_turn>
<start_of_turn>model
허리가 안 좋으시군요. 그런 경우에는 데드리프트 중량을 조절하여 안전하게 운동하는 것이 중요합니다. 531 프로그램을 기반으로 주차별로 추천 중량을 안내해드릴게요.

먼저, 당신의 1RM(들 수 있는 최대 무게)을 기준으로 TM(실제 1RM의 90%)을 계산해보겠습니다.

- **스쿼트 TM**: 215kg × 0.9 = 193.5kg
- **밀리터리 프레스 TM**: 103.2kg × 0.9 = 92.88kg
- **벤치 프레스 TM**: 147.5kg × 0.9 = 132.75kg
- **데드리프트 TM**: 222.5kg × 0.9 = 200.25kg

이제 주차별로 추천 중량을 안내해드리겠습니다. 데드리프트는 허리를 고려하여 중량을 약간 낮춰서 설정하겠습니다.

### 1주차 (TM의 65%, 75%, 85%):
- **스쿼트**: 5회 125kg, 5회 145kg, 5회 이상 165kg
- **밀리터리 프레스**: 5회 65kg, 5회 75kg, 5회 이상 85kg
- **벤치 프레스**: 5회 95kg, 5회 110kg, 5회 이상 125kg
- **데드리프트**: 5회 130kg, 5회 150kg, 5회 이상 170kg (허리를 고려하여 중량을 조절했습니다)

### 2주차 (TM의 70%, 80%, 90%):
- **스쿼트**: 3회 135kg, 3회 155kg, 3회 이상 175kg
- **밀리터리 프레스**: 3회 70kg, 3회 80kg, 3회 이상 90kg
- **벤치 프레스**: 3회 105kg, 3회 120kg, 3회 이상 135kg
- **데드리프트**: 3회 140kg,

## 4. 모델 학습

In [ ]:
trainer = kd.train.Trainer(
    seed=42,  # The seed of enlightenment
    workdir='/tmp/ckpts',  # TODO(epot): Make the workdir optional by default
    # Dataset
    train_ds=ds,
    # Model
    model=model,
    init_transform=MyInitTransform(), # init_transform=init_transform : 첫 번째인 경우만 # init_transform=MyInitTransform()
    # Training parameters
    num_train_steps=300,
    train_losses={
        "loss": kd.losses.SoftmaxCrossEntropyWithIntLabels(
            logits="preds.logits",
            labels="batch.target",
            mask="batch.loss_mask",
        ),
    },
    optimizer=optimizer,
)

In [ ]:
state, aux = trainer.train()

Disabling pygrain multi-processing (unsupported in colab).
Starting training loop at step 0


train:   0%|          | 0/301 [00:00<?, ?it/s]

In [ ]:
sampler = gm.text.ChatSampler(
    model=model,
    params=state.params,
    tokenizer=tokenizer,
)

## 5. 샘플 출력 확인

In [ ]:
sampler.chat("나는 웨이트 트레이닝 2년차야. 531 운동 프로그램 루틴을 작성해줘. {'성별': '남성', '몸무게': 121.05, 'squat_1RM': 100, 'press_1RM': 35, 'bench_press_1RM': 50, 'deadlift_1RM': 110}")

'531 프로그램은 웨이트 트레이닝을 기반으로 한 루틴입니다. 이 프로그램은 점진적으로 중량을 증가시켜가며, 4주 주기로 구성되어 있습니다.\n\n먼저, 당신의 1RM(들 수 있는 최대 무게)을 기준으로 TM(실제 1RM의 90%)을 계산해보겠습니다.\n\n- **스쿼트 TM**: 100kg × 0.9 = 90kg\n- **밀리터리TrackAngle(밀리터리) 1RM**: 35kg × 0.9 = 31.5kg\n- **벤치 프레스 1RM**: 50kg × 0.9 = 45kg\n- **데드리프트 1RM**: 110kg × 0.9 = 99kg\n\n이제 주차별로 추천 중량을 안내해드리겠습니다.\n\n### 1주차 (TM의 65%, 75%, 85%):\n- **스쿼트**: 5회 57.5kg, 5회 67.5kg, 5회 이상 77.5kg\n- **밀리터리TrackAngle**: 5회 22.5kg, 5회 30kg, 5회 이상 35kg\n- **벤치 프레스**: 5회 30kg, 5회 35kg, 5회 이상 40kg\n- **데드리프트**: 5회 70kg, 5회 80kg, 5회 이상 90kg\n\n### 2주차 (TM의 70%, 80%, 90%):\n- **스쿼트**: 3회 62.5kg, 3회 72.5kg, 3회 이상 82.5kg\n- **밀리터리TrackAngle**: 3회 25kg, 3회 30kg, 3회 이상 35kg\n- **벤치 프레스**: 3회 32.5kg, 3회 37.5kg, 3회 이상 42.5kg\n- **데드리프트**: 3회 75kg, 3회 85kg, 3회 이상 95kg\n\n### 3주차 (TM의 75%, 85%, 95%):\n- **스쿼트**: 5회 67.5kg, 3회 77.5kg, 1회 이상 87.5kg\n- **밀리터리TrackAngle**: 5회 27.5kg, 3회 32.5kg, 1회 이상 35kg\n- **벤치 프레스**: 5회 35kg, 3회 40kg, 1회 이상 45kg\n- **데드리프트**: 5회 80kg, 3회 90kg, 1회 이상 100kg\

## 6. 모델 파라미터 저장

In [ ]:
gm.ckpts.save_params(state.params, f'/content/drive/MyDrive/Gemma3-lora{step}')